# EDA interactivo curado para el proyecto de churn

Esta versión está más enfocada a la **defensa** y a la futura app de **Streamlit**. He quitado parte del ruido exploratorio inicial y he dejado los gráficos que mejor cuentan la historia del churn: **tipo de suscripción, engagement, fricción, segmentos y geografía**.

## 1. Imports y carga de datos

In [ ]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)


In [ ]:

def load_data():
    candidate_paths = [
        "data/raw/train.csv",
        "dataproject/train.csv",
        "train.csv"
    ]
    for path in candidate_paths:
        try:
            return pd.read_csv(path)
        except FileNotFoundError:
            continue
    raise FileNotFoundError(
        "No se encontró el archivo train.csv. Prueba con data/raw/train.csv, dataproject/train.csv o train.csv."
    )

df = load_data().copy()
df.head()


## 2. Funciones de preparación

Estas funciones dejan el dataset listo para el EDA sin mezclar todavía la parte de modelado.

In [ ]:

def clean_streaming_churn_df(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    df = df.copy()

    # Variables derivadas más interpretables
    if "signup_date" in df.columns:
        # En este dataset signup_date aparece en negativo, así que lo convertimos a antigüedad positiva.
        df["tenure_days"] = -df["signup_date"]
        df["tenure_months"] = df["tenure_days"] / 30

    if "weekly_hours" in df.columns and "weekly_songs_played" in df.columns:
        df["songs_per_hour"] = np.where(
            df["weekly_hours"] > 0,
            df["weekly_songs_played"] / df["weekly_hours"],
            np.nan
        )

    if "song_skip_rate" in df.columns:
        df["high_skip_user"] = (df["song_skip_rate"] >= 0.7).astype(int)

    return df


def add_eda_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "age" in df.columns:
        df["age_group"] = pd.cut(
            df["age"],
            bins=[18, 25, 35, 50, 65, 80],
            labels=["18-24", "25-34", "35-49", "50-64", "65-79"],
            include_lowest=True
        )

    if "weekly_hours" in df.columns:
        df["weekly_hours_bin"] = pd.cut(
            df["weekly_hours"],
            bins=[0, 5, 10, 20, 30, 40, 50],
            labels=["0-5", "5-10", "10-20", "20-30", "30-40", "40-50"],
            include_lowest=True
        )

    if "song_skip_rate" in df.columns:
        df["skip_rate_bin"] = pd.cut(
            df["song_skip_rate"],
            bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
            labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"],
            include_lowest=True
        )

    if {"subscription_type", "customer_service_inquiries"}.issubset(df.columns):
        conditions = [
            (df["subscription_type"] == "Free") & (df["customer_service_inquiries"] == "High"),
            (df["subscription_type"] == "Free"),
            (df["customer_service_inquiries"] == "High")
        ]
        choices = ["Very High Risk Segment", "High Risk Segment", "Medium Risk Segment"]
        df["risk_segment_rule"] = np.select(conditions, choices, default="Lower Risk Segment")

    return df


df = clean_streaming_churn_df(df, is_train=True)
df = add_eda_features(df)
df.head()


## 3. Vista general rápida

In [ ]:

def basic_overview(df: pd.DataFrame, name: str = "DataFrame") -> None:
    print(f"Shape de {name}: {df.shape}")
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    print(df.isna().sum())
    if "churned" in df.columns:
        print(f"\nTasa global de churn: {df['churned'].mean():.4f}")


basic_overview(df, name="train_clean")


## 4. Funciones de visualización interactivas

In [ ]:

def churn_rate(df, target="churned"):
    return df[target].mean() * 100


def plot_global_churn(df, target="churned"):
    churn_pct = df[target].mean() * 100
    no_churn_pct = 100 - churn_pct

    plot_df = pd.DataFrame({
        "Estado": ["No churn", "Churn"],
        "Porcentaje": [no_churn_pct, churn_pct]
    })

    fig = px.bar(
        plot_df,
        x="Estado",
        y="Porcentaje",
        text="Porcentaje",
        title="Distribución global del churn"
    )

    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig.update_layout(
        yaxis_title="Porcentaje",
        xaxis_title="",
        title_x=0.5
    )
    fig.show()


def plot_churn_by_category(df, category_col, target="churned", sort_desc=True, title=None):
    plot_df = (
        df.groupby(category_col)[target]
        .mean()
        .reset_index()
        .assign(churn_pct=lambda x: x[target] * 100)
        [[category_col, "churn_pct"]]
    )

    if sort_desc:
        plot_df = plot_df.sort_values("churn_pct", ascending=False)

    fig = px.bar(
        plot_df,
        x=category_col,
        y="churn_pct",
        text="churn_pct",
        title=title if title else f"Tasa de churn por {category_col}"
    )

    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig.update_layout(
        yaxis_title="Churn (%)",
        xaxis_title=category_col,
        title_x=0.5
    )
    fig.show()

    return plot_df


def plot_churn_by_numeric_bins(df, numeric_col, bin_col, target="churned", title=None):
    plot_df = (
        df.groupby(bin_col, observed=False)[target]
        .mean()
        .reset_index()
        .assign(churn_pct=lambda x: x[target] * 100)
        [[bin_col, "churn_pct"]]
    )

    fig = px.bar(
        plot_df,
        x=bin_col,
        y="churn_pct",
        text="churn_pct",
        title=title if title else f"Tasa de churn por intervalos de {numeric_col}"
    )

    fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
    fig.update_layout(
        yaxis_title="Churn (%)",
        xaxis_title=numeric_col,
        title_x=0.5
    )
    fig.show()

    return plot_df


def plot_churn_heatmap(df, row_col, col_col, target="churned", title=None):
    heatmap_df = (
        df.groupby([row_col, col_col])[target]
        .mean()
        .reset_index()
        .assign(churn_pct=lambda x: x[target] * 100)
        .pivot(index=row_col, columns=col_col, values="churn_pct")
    )

    fig = px.imshow(
        heatmap_df,
        text_auto=".2f",
        aspect="auto",
        color_continuous_scale="Reds",
        title=title if title else f"Heatmap de churn (%) entre {row_col} y {col_col}",
        labels=dict(color="Churn (%)")
    )
    fig.update_layout(title_x=0.5)
    fig.show()

    return heatmap_df


def plot_distribution_by_churn(df, numeric_col, target="churned", nbins=30, title=None):
    df_copy = df.copy()
    df_copy["churn_label"] = df_copy[target].map({0: "No churn", 1: "Churn"})

    fig = px.histogram(
        df_copy,
        x=numeric_col,
        color="churn_label",
        barmode="overlay",
        nbins=nbins,
        opacity=0.6,
        title=title if title else f"Distribución de {numeric_col} según churn"
    )

    fig.update_layout(
        xaxis_title=numeric_col,
        yaxis_title="Frecuencia",
        title_x=0.5
    )
    fig.show()


## 5. Gráficos principales del EDA

Estos son los gráficos que más merecen la pena para la defensa y para reutilizar luego en Streamlit.

In [ ]:
print(f"Tasa global de churn: {churn_rate(df):.2f}%")
plot_global_churn(df)

### 5.1 Tipo de suscripción

Este gráfico suele ser uno de los más fuertes del proyecto: permite ver si el compromiso económico con la plataforma se asocia con el churn.

In [ ]:
sub_table = plot_churn_by_category(df, "subscription_type", title="Tasa de churn por tipo de suscripción")
sub_table

### 5.2 Fricción del usuario

`customer_service_inquiries` captura muy bien una dimensión de fricción o mala experiencia de usuario.

In [ ]:
cs_table = plot_churn_by_category(df, "customer_service_inquiries", title="Tasa de churn por incidencias / consultas")
cs_table

### 5.3 Engagement semanal

Las horas semanales son una de las variables numéricas más potentes del dataset.

In [ ]:
wh_table = plot_churn_by_numeric_bins(df, "weekly_hours", "weekly_hours_bin", title="Tasa de churn por intervalos de weekly_hours")
wh_table

### 5.4 Pausas en la suscripción

Las pausas actúan como señal de inestabilidad. Es un gráfico sencillo pero muy explicable en la defensa.

In [ ]:
pauses_table = plot_churn_by_category(df, "num_subscription_pauses", sort_desc=False, title="Tasa de churn por número de pausas")
pauses_table

### 5.5 Song skip rate

Este gráfico refuerza la idea de menor ajuste o menor satisfacción con el contenido.

In [ ]:
skip_table = plot_churn_by_numeric_bins(df, "song_skip_rate", "skip_rate_bin", title="Tasa de churn por intervalos de song_skip_rate")
skip_table

### 5.6 Edad por tramos

La edad no suele comportarse de forma lineal, así que por tramos se interpreta mejor.

In [ ]:
age_table = plot_churn_by_numeric_bins(df, "age", "age_group", title="Tasa de churn por grupos de edad")
age_table

### 5.7 Interacción clave: plan × incidencias

Este heatmap es especialmente útil porque combina dos de las variables más importantes del dataset.

In [ ]:
segment_heatmap = plot_churn_heatmap(
    df,
    "subscription_type",
    "customer_service_inquiries",
    title="Heatmap de churn (%) entre subscription_type y customer_service_inquiries"
)
segment_heatmap

### 5.8 Distribución de `weekly_hours` según churn

Dejo un gráfico de distribución para ilustrar visualmente la separación entre churn / no churn, pero sin sobrecargar con muchos histogramas o boxplots redundantes.

In [ ]:
plot_distribution_by_churn(df, "weekly_hours", nbins=30, title="Distribución de weekly_hours según churn")

## 6. Geografía

Aquí dejo una función automática: si `location` contiene estados de EE. UU., dibuja un **choropleth**; si contiene regiones, dibuja un **mapa de burbujas** por región.

In [ ]:

def plot_location_map(df, location_col="location", target_col="churned"):
    values = df[location_col].dropna().astype(str)
    unique_values = set(values.unique())

    us_state_codes = {
        "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS",
        "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY",
        "NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV",
        "WI","WY"
    }

    region_labels = {"West", "Midwest", "South", "Northeast"}

    # Caso 1: abreviaturas de estados
    if unique_values.issubset(us_state_codes):
        plot_df = (
            df.groupby(location_col)
            .agg(
                n_users=(location_col, "size"),
                churn_rate=(target_col, "mean")
            )
            .reset_index()
        )

        fig = px.choropleth(
            plot_df,
            locations=location_col,
            locationmode="USA-states",
            color="n_users",
            scope="usa",
            color_continuous_scale="Burg",
            hover_name=location_col,
            hover_data={"n_users": True, "churn_rate": ":.2%"},
            title="Distribución geográfica de usuarios por estado"
        )

        fig.update_layout(
            title_x=0.5,
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(l=20, r=20, t=60, b=20)
        )
        fig.show()
        return plot_df

    # Caso 2: regiones
    elif unique_values.issubset(region_labels):
        region_counts = (
            df.groupby(location_col)
            .agg(
                n_users=(location_col, "size"),
                churn_rate=(target_col, "mean")
            )
            .reset_index()
        )

        region_coords = {
            "West": {"lat": 39, "lon": -120},
            "Midwest": {"lat": 41, "lon": -93},
            "South": {"lat": 33, "lon": -86},
            "Northeast": {"lat": 42, "lon": -74}
        }

        region_counts["lat"] = region_counts[location_col].map(lambda x: region_coords.get(x, {}).get("lat"))
        region_counts["lon"] = region_counts[location_col].map(lambda x: region_coords.get(x, {}).get("lon"))

        fig = px.scatter_geo(
            region_counts,
            lat="lat",
            lon="lon",
            size="n_users",
            color="n_users",
            scope="usa",
            hover_name=location_col,
            hover_data={"n_users": True, "churn_rate": ":.2%"},
            color_continuous_scale="Burg",
            title="Distribución geográfica de usuarios por región"
        )

        fig.update_layout(
            title_x=0.5,
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(l=20, r=20, t=60, b=20)
        )
        fig.show()
        return region_counts

    else:
        print("La columna location no parece contener ni abreviaturas de estados ni regiones estándar.")
        print("Valores de ejemplo:", values.unique()[:10])
        return None


In [ ]:
geo_table = plot_location_map(df, location_col="location", target_col="churned")
geo_table

## 7. Tablas resumen útiles para la defensa y para Streamlit

In [ ]:

segment_summary = (
    df.groupby(["subscription_type", "customer_service_inquiries"], as_index=False)
    .agg(
        churn_rate=("churned", "mean"),
        avg_weekly_hours=("weekly_hours", "mean"),
        avg_skip_rate=("song_skip_rate", "mean"),
        avg_pauses=("num_subscription_pauses", "mean"),
        n_customers=("churned", "size")
    )
    .sort_values("churn_rate", ascending=False)
)

segment_summary


## 8. Conclusiones recomendadas para contar en la defensa


### Conclusiones preliminares del EDA

A partir del análisis exploratorio, las señales más fuertes de churn parecen concentrarse en tres grandes bloques:

1. **Tipo de usuario**  
   El tipo de suscripción marca diferencias muy claras de churn, especialmente entre usuarios **Free** y planes de pago.

2. **Engagement**  
   Variables como `weekly_hours` y `song_skip_rate` sugieren que los usuarios menos activos o con peor interacción con el contenido presentan mayor riesgo de abandono.

3. **Fricción / inestabilidad**  
   Variables como `customer_service_inquiries` y `num_subscription_pauses` parecen actuar como señales tempranas de churn.

Además, el heatmap por segmentos muestra que algunas combinaciones de variables, como determinados planes junto con alta fricción, generan perfiles especialmente vulnerables.


## 9. Qué he quitado respecto al notebook original

- parte del bloque de outliers demasiado largo para la historia principal
- gráficos redundantes de la misma variable en varios formatos
- exploraciones con poco valor narrativo para la defensa
- análisis geográfico muy disperso en varias subsecciones

Si quieres, este notebook ya queda mucho mejor como base para **Streamlit** y para la **presentación oral**.